In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, random
import numpy as np
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE      = '/content/drive/MyDrive/LegalDocAnalyzer'
REPO_PATH = '/content/legal-doc-analyzer'
sys.path.append(f'{BASE}/src')


LABEL_LIST = [
    "O",
    "B-PARTY",       "I-PARTY",
    "B-DATE",        "I-DATE",
    "B-AMOUNT",      "I-AMOUNT",
    "B-TERM",        "I-TERM",
    "B-JURISDICTION","I-JURISDICTION"
]

LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

print(" Setup complete")
print(f"\nLabel list ({len(LABEL_LIST)} labels):")
for i, label in enumerate(LABEL_LIST):
    print(f"  {i:2d}  {label}")

Mounted at /content/drive
✅ Setup complete

Label list (11 labels):
   0  O
   1  B-PARTY
   2  I-PARTY
   3  B-DATE
   4  I-DATE
   5  B-AMOUNT
   6  I-AMOUNT
   7  B-TERM
   8  I-TERM
   9  B-JURISDICTION
  10  I-JURISDICTION


In [22]:
with open(f'{BASE}/data/raw/cuad_mapped.json') as f:
    raw_data = json.load(f)
print(f" Reloaded raw_data: {len(raw_data)} original QA examples")


 Reloaded raw_data: 5686 original QA examples


In [23]:
import re

ENTITY_MAP = {
    "Parties"                            : "PARTY",
    "Agreement Date"                     : "DATE",
    "Effective Date"                     : "DATE",
    "Expiration Date"                    : "DATE",
    "Renewal Term"                       : "TERM",
    "Notice Period To Terminate Renewal" : "TERM",
    "Governing Law"                      : "JURISDICTION",
    "Most Favored Nation"                : "TERM",
    "Non-Compete"                        : "TERM",
    "Exclusivity"                        : "TERM",
    "No-Solicit Of Customers"            : "TERM",
    "Liquidated Damages"                 : "AMOUNT",
    "Warranty Duration"                  : "TERM",
    "Insurance"                          : "AMOUNT",
    "Minimum Commitment"                 : "AMOUNT",
    "Volume Restriction"                 : "AMOUNT",
    "Price Restrictions"                 : "AMOUNT",
    "IP Ownership Assignment"            : "TERM",
    "License Grant"                      : "TERM",
    "Revenue/Profit Sharing"             : "AMOUNT",
    "Cap On Liability"                   : "AMOUNT",
    "Uncapped Liability"                 : "AMOUNT",
    "Third Party Beneficiary"            : "PARTY",
    "Post-Termination Services"          : "TERM",
    "Audit Rights"                       : "TERM",
    "Unlimited/All-You-Can-Eat-License"  : "TERM",
    "Irrevocable Or Perpetual License"   : "TERM",
    "Source Code Escrow"                 : "TERM",
    "Change Of Control"                  : "TERM",
    "Termination For Convenience"        : "TERM",
    "Covenant Not To Sue"                : "TERM",
    "Competitive Restriction Exception"  : "TERM",
    "No-Solicit Of Employees"            : "TERM",
    "Affiliate License-Licensee"         : "PARTY",
    "Joint Ip Ownership"                 : "TERM",
    "Non-Disparagement"                  : "TERM",
    "Affiliate License-Licensor"         : "PARTY",
}

ENTITY_MAP_LOWER = {k.lower(): v for k, v in ENTITY_MAP.items()}

os.makedirs(f'{BASE}/src', exist_ok=True)
with open(f'{BASE}/src/entity_map.json', 'w') as f:
    json.dump(ENTITY_MAP, f, indent=2)
print(f"✅ entity_map.json saved ({len(ENTITY_MAP)} mappings)")

MAX_LEN = 320
STRIDE  = 128


def lookup_entity(question, entity_label):
    q = question.strip().lower()
    if q in ENTITY_MAP_LOWER:
        return ENTITY_MAP_LOWER[q]
    e = entity_label.strip().lower()
    if e in ENTITY_MAP_LOWER:
        return ENTITY_MAP_LOWER[e]
    return None


def tokenize_context(context):
    word_pattern = re.compile(r"\w+|[^\w\s]")
    return [(m.group(), m.start(), m.end())
            for m in word_pattern.finditer(context)]


def build_iob2(word_spans, answer_start, answer_end, entity_label):
    tokens   = []
    ner_tags = []
    entity_started = False

    for word, w_start, w_end in word_spans:
        tokens.append(word)
        overlaps = (w_start < answer_end and w_end > answer_start)

        if overlaps:
            if not entity_started:
                ner_tags.append(LABEL2ID[f"B-{entity_label}"])
                entity_started = True
            else:
                ner_tags.append(LABEL2ID[f"I-{entity_label}"])
        else:
            entity_started = False
            ner_tags.append(LABEL2ID["O"])

    if not any(t != LABEL2ID["O"] for t in ner_tags):
        return None, None

    return tokens, ner_tags


def check_iob2(tags):
    prev = "O"
    for t in tags:
        label = ID2LABEL[t]
        if label.startswith("I-"):
            entity = label[2:]
            if prev not in [f"B-{entity}", f"I-{entity}"]:
                return False
        prev = label
    return True


def slide_and_collect(tokens, ner_tags, entity_label):
    chunks = []
    if len(tokens) <= MAX_LEN:
        if any(t != LABEL2ID["O"] for t in ner_tags):
            chunks.append({"tokens": tokens, "ner_tags": ner_tags,
                           "entity": entity_label})
    else:
        for start in range(0, len(tokens), MAX_LEN - STRIDE):
            end          = start + MAX_LEN
            chunk_tok    = tokens[start:end]
            chunk_tag    = ner_tags[start:end]
            if any(t != LABEL2ID["O"] for t in chunk_tag):
                if check_iob2(chunk_tag):
                    chunks.append({"tokens": chunk_tok,
                                   "ner_tags": chunk_tag,
                                   "entity": entity_label})
    return chunks


print("Converting QA format → IOB2 format with sliding windows...")
print(f"Total examples: {len(raw_data)}")

converted = []
stats = Counter({
    "total"           : 0,
    "converted"       : 0,
    "chunks_added"    : 0,
    "no_answer"       : 0,
    "unknown_question": 0,
    "conversion_fail" : 0,
    "iob2_invalid"    : 0,
})

for idx, example in enumerate(raw_data):
    stats["total"] += 1

    question     = example.get("question",     "")
    entity_label = example.get("entity_label", "")
    has_answer   = example.get("has_answer",   False)
    answers      = example.get("answers",      {})
    context      = example.get("context",      "")

    mapped_entity = lookup_entity(question, entity_label)

    if not mapped_entity:
        stats["unknown_question"] += 1
        continue

    if not has_answer:
        stats["no_answer"] += 1
        continue

    answer_texts  = answers.get("text",         [])
    answer_starts = answers.get("answer_start", [])

    if not answer_texts or not answer_starts:
        stats["no_answer"] += 1
        continue

    word_spans = tokenize_context(context)
    if not word_spans:
        stats["conversion_fail"] += 1
        continue

    tokens   = None
    ner_tags = None

    for answer_text, answer_start in zip(answer_texts, answer_starts):
        answer_end = answer_start + len(answer_text)
        t, g = build_iob2(word_spans, answer_start, answer_end, mapped_entity)
        if t is not None:
            tokens   = t
            ner_tags = g
            break

    if tokens is None:
        stats["conversion_fail"] += 1
        continue

    chunks = slide_and_collect(tokens, ner_tags, mapped_entity)

    if not chunks:
        stats["conversion_fail"] += 1
        continue

    converted.extend(chunks)
    stats["converted"]    += 1
    stats["chunks_added"] += len(chunks)

    if (idx + 1) % 500 == 0:
        print(f"  Processed {idx+1}/{len(raw_data)} → "
              f"{stats['chunks_added']} chunks so far")

print(f"\n=== CONVERSION RESULTS ===")
print(f"  Total input        : {stats['total']}")
print(f"  Examples converted : {stats['converted']}")
print(f"  Total chunks       : {stats['chunks_added']}")
print(f"  No answer          : {stats['no_answer']}")
print(f"  Unknown question   : {stats['unknown_question']}")
print(f"  Conversion fail    : {stats['conversion_fail']}")
print(f"  IOB2 invalid       : {stats['iob2_invalid']}")

avg_len = sum(len(ex["tokens"]) for ex in converted) / max(len(converted), 1)
print(f"\n  Average token length : {avg_len:.1f}  (ideal: 80–220)")
print(f"  Status               : {'✅' if 80 <= avg_len <= 220 else '⚠️'}")

print(f"\n=== ENTITY DISTRIBUTION ===")
entity_counts = Counter(ex["entity"] for ex in converted)
if entity_counts:
    max_count = max(entity_counts.values())
    for entity, count in sorted(entity_counts.items(), key=lambda x: -x[1]):
        bar = "█" * int(count / max_count * 30)
        pct = 100 * count / len(converted)
        print(f"  {entity:15s}: {count:5d}  {bar}  ({pct:.1f}%)")

print(f"\n=== SAMPLE CONVERTED EXAMPLES ===")
shown = set()
for ex in converted:
    entity = ex["entity"]
    if entity not in shown:
        entity_tokens = [
            ex["tokens"][i]
            for i, t in enumerate(ex["ner_tags"])
            if t != LABEL2ID["O"]
        ]
        ctx = " ".join(ex["tokens"][:10])
        print(f"\n  Entity  : {entity}")
        print(f"  Found   : '{' '.join(entity_tokens[:8])}'")
        print(f"  Context : '{ctx}...'")
        shown.add(entity)
    if len(shown) == 5:
        break

raw_data = [{"tokens": ex["tokens"], "ner_tags": ex["ner_tags"]}
            for ex in converted]

print(f"\n✅ raw_data ready   : {len(raw_data)} examples")
status = "✅ good" if len(raw_data) >= 3000 else "⚠️ lower than expected"
print(f"   Status           : {status}  (expected: 3000–6000)")

✅ entity_map.json saved (37 mappings)
Converting QA format → IOB2 format with sliding windows...
Total examples: 5686
  Processed 500/5686 → 698 chunks so far
  Processed 1000/5686 → 1384 chunks so far
  Processed 1500/5686 → 2070 chunks so far
  Processed 2000/5686 → 2776 chunks so far
  Processed 2500/5686 → 3481 chunks so far
  Processed 3000/5686 → 4191 chunks so far
  Processed 3500/5686 → 4908 chunks so far
  Processed 4000/5686 → 5618 chunks so far
  Processed 4500/5686 → 6301 chunks so far
  Processed 5000/5686 → 7015 chunks so far
  Processed 5500/5686 → 7697 chunks so far

=== CONVERSION RESULTS ===
  Total input        : 5686
  Examples converted : 5176
  Total chunks       : 7946
  No answer          : 0
  Unknown question   : 510
  Conversion fail    : 0
  IOB2 invalid       : 0

  Average token length : 318.3  (ideal: 80–220)
  Status               : ⚠️

=== ENTITY DISTRIBUTION ===
  TERM           :  3353  ██████████████████████████████  (42.2%)
  DATE           :  1698 

In [24]:
print("Running IOB2 validation on all examples...")

clean_tokens = []
clean_tags   = []
stats = {
    "total"             : len(raw_data),
    "kept"              : 0,
    "fixed"             : 0,
    "skipped_empty"     : 0,
    "skipped_all_o"     : 0,
    "skipped_too_short" : 0,
    "skipped_too_long"  : 0,
}

def validate_and_fix(tokens, tags):
    fixed  = list(tags)
    errors = []

    for i, tag in enumerate(tags):
        label = ID2LABEL[tag]
        if label.startswith("I-"):
            entity  = label[2:]
            b_label = f"B-{entity}"
            i_label = f"I-{entity}"
            if i == 0:
                fixed[i] = LABEL2ID[b_label]
                errors.append(f"pos {i}: {label} at start → {b_label}")
            elif ID2LABEL[fixed[i-1]] not in [b_label, i_label]:
                fixed[i] = LABEL2ID[b_label]
                errors.append(f"pos {i}: {label} after {ID2LABEL[fixed[i-1]]} → {b_label}")

    return fixed, len(errors) > 0

for example in raw_data:
    tokens = example["tokens"]
    tags   = example["ner_tags"]

    if len(tokens) == 0:
        stats["skipped_empty"] += 1
        continue

    if len(tokens) < 5:
        stats["skipped_too_short"] += 1
        continue

    if len(tokens) > 320:
        stats["skipped_too_long"] += 1
        continue

    if all(t == LABEL2ID["O"] for t in tags):
        stats["skipped_all_o"] += 1
        continue

    fixed_tags, was_fixed = validate_and_fix(tokens, tags)
    if was_fixed:
        stats["fixed"] += 1

    clean_tokens.append(tokens)
    clean_tags.append(fixed_tags)
    stats["kept"] += 1

print(f"\n=== VALIDATION RESULTS ===")
print(f"  Total input      : {stats['total']}")
print(f"  Kept             : {stats['kept']}")
print(f"  Fixed (IOB2)     : {stats['fixed']}")
print(f"  Skipped (empty)  : {stats['skipped_empty']}")
print(f"  Skipped (all-O)  : {stats['skipped_all_o']}")
print(f"  Skipped (short)  : {stats['skipped_too_short']}")
print(f"  Skipped (long)   : {stats['skipped_too_long']}")

print(f"\n=== ENTITY DISTRIBUTION (clean data) ===")
counts = Counter()
for tags in clean_tags:
    for tag in tags:
        label = ID2LABEL[tag]
        if label.startswith("B-"):
            counts[label[2:]] += 1

max_count = max(counts.values())
for entity, count in sorted(counts.items(), key=lambda x: -x[1]):
    bar = "=" * int(count / max_count * 30)
    pct = 100 * count / sum(counts.values())
    print(f"  {entity:15s}: {count:5d}  {bar}  ({pct:.1f}%)")

status = "✅" if stats['kept'] >= 5000 else "⚠️"
print(f"\n  Clean examples : {stats['kept']}  {status}")

Running IOB2 validation on all examples...

=== VALIDATION RESULTS ===
  Total input      : 7946
  Kept             : 7946
  Fixed (IOB2)     : 0
  Skipped (empty)  : 0
  Skipped (all-O)  : 0
  Skipped (short)  : 0
  Skipped (long)   : 0

=== ENTITY DISTRIBUTION (clean data) ===
  TERM           :  3353  ==============================  (42.2%)
  DATE           :  1698  ===============  (21.4%)
  AMOUNT         :  1473  =============  (18.5%)
  JURISDICTION   :   728  ======  (9.2%)
  PARTY          :   694  ======  (8.7%)

  Clean examples : 7946  ✅


In [25]:
from sklearn.model_selection import train_test_split

def get_rarest_entity(tags):
    rarity_order = {
        "JURISDICTION": 0,
        "PARTY"       : 1,
        "AMOUNT"      : 2,
        "DATE"        : 3,
        "TERM"        : 4,
    }
    found = set()
    for tag in tags:
        label = ID2LABEL[tag]
        if label.startswith("B-"):
            found.add(label[2:])
    if not found:
        return "O_ONLY"
    return min(found, key=lambda e: rarity_order.get(e, 5))

strat_keys = [get_rarest_entity(tags) for tags in clean_tags]

print("Stratification key distribution:")
key_counts = Counter(strat_keys)
for key, count in sorted(key_counts.items()):
    print(f"  {key:15s}: {count}")

indices = list(range(len(clean_tokens)))

train_idx, temp_idx = train_test_split(
    indices,
    test_size    = 0.30,
    random_state = SEED,
    stratify     = [strat_keys[i] for i in indices]
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size    = 0.50,
    random_state = SEED,
    stratify     = [strat_keys[i] for i in temp_idx]
)

train_tokens = [clean_tokens[i] for i in train_idx]
train_tags   = [clean_tags[i]   for i in train_idx]
val_tokens   = [clean_tokens[i] for i in val_idx]
val_tags     = [clean_tags[i]   for i in val_idx]
test_tokens  = [clean_tokens[i] for i in test_idx]
test_tags    = [clean_tags[i]   for i in test_idx]

print(f"\nSPLIT SIZES ")
print(f"  Train : {len(train_tokens)} ({100*len(train_tokens)/len(clean_tokens):.0f}%)")
print(f"  Val   : {len(val_tokens)}   ({100*len(val_tokens)/len(clean_tokens):.0f}%)")
print(f"  Test  : {len(test_tokens)}  ({100*len(test_tokens)/len(clean_tokens):.0f}%) ← FROZEN")

print(f"\nJURISDICTION ACROSS SPLITS")
for name, tags_list in [("Train", train_tags),
                         ("Val",   val_tags),
                         ("Test",  test_tags)]:
    j = sum(1 for tags in tags_list
            for tag in tags
            if ID2LABEL[tag] == "B-JURISDICTION")
    status = "OK" if j >= 20 else "very few"
    print(f"  {name:5s}: {j:4d} JURISDICTION examples  {status}")

print(f"\nPARTY ACROSS SPLITS")
for name, tags_list in [("Train", train_tags),
                         ("Val",   val_tags),
                         ("Test",  test_tags)]:
    p = sum(1 for tags in tags_list
            for tag in tags
            if ID2LABEL[tag] == "B-PARTY")
    status = "OK" if p >= 20 else " very few"
    print(f"  {name:5s}: {p:4d} PARTY examples  {status}")

Stratification key distribution:
  AMOUNT         : 1473
  DATE           : 1698
  JURISDICTION   : 728
  PARTY          : 694
  TERM           : 3353

SPLIT SIZES 
  Train : 5562 (70%)
  Val   : 1192   (15%)
  Test  : 1192  (15%) ← FROZEN

JURISDICTION ACROSS SPLITS
  Train:  510 JURISDICTION examples  OK
  Val  :  109 JURISDICTION examples  OK
  Test :  109 JURISDICTION examples  OK

PARTY ACROSS SPLITS
  Train:  486 PARTY examples  OK
  Val  :  104 PARTY examples  OK
  Test :  104 PARTY examples  OK


In [26]:
print(" ENTITY COUNTS BEFORE OVERSAMPLING ")
before = Counter()
for tags in train_tags:
    for tag in tags:
        label = ID2LABEL[tag]
        if label.startswith("B-"):
            before[label[2:]] += 1

max_before = max(before.values())
for entity, count in sorted(before.items(), key=lambda x: -x[1]):
    bar = "=" * int(count / max_before * 30)
    print(f"  {entity:15s}: {count:5d}  {bar}")

groups = {"PARTY": [], "DATE": [], "AMOUNT": [], "TERM": [], "JURISDICTION": []}

for tokens, tags in zip(train_tokens, train_tags):
    rarest = get_rarest_entity(tags)
    if rarest in groups:
        groups[rarest].append((tokens, tags))

counts_list  = [len(g) for g in groups.values() if len(g) > 0]
MEDIAN_COUNT = int(np.median(counts_list))
MAX_MULT     = 2.0

print(f"\n  Median count   : {MEDIAN_COUNT}")
print(f"  Max multiplier : {MAX_MULT}x")
print(f"\n OVERSAMPLING PLAN ")

os_tokens = []
os_tags   = []

for entity, examples in groups.items():
    current = len(examples)
    if current == 0:
        continue

    for t, g in examples:
        os_tokens.append(t)
        os_tags.append(g)

    max_allowed  = int(current * MAX_MULT)
    target       = min(MEDIAN_COUNT, max_allowed)
    needed       = max(0, target - current)

    if needed > 0:
        extras = random.choices(examples, k=needed)
        for t, g in extras:
            os_tokens.append(t)
            os_tags.append(g)

    final = current + needed
    mult  = round(final / current, 1)
    risk  = "OK" if mult <= 2.0 else "NO"
    print(f"  {entity:15s}: {current:4d} → {final:4d}  ({mult}x)  {risk}")

combined = list(zip(os_tokens, os_tags))
random.shuffle(combined)
os_tokens, os_tags = zip(*combined)
os_tokens = list(os_tokens)
os_tags   = list(os_tags)

print(f"\n ENTITY COUNTS AFTER OVERSAMPLING ")
after = Counter()
for tags in os_tags:
    for tag in tags:
        label = ID2LABEL[tag]
        if label.startswith("B-"):
            after[label[2:]] += 1

max_after = max(after.values())
for entity, count in sorted(after.items(), key=lambda x: -x[1]):
    bar = "=" * int(count / max_after * 30)
    print(f"  {entity:15s}: {count:5d}  {bar}")

print(f"\n  Total training examples: {len(os_tokens)}")

#  apply suggestion
train_data = [{"tokens": t, "ner_tags": g}
              for t, g in zip(os_tokens, os_tags)]

val_data   = [{"tokens": t, "ner_tags": g}
              for t, g in zip(val_tokens, val_tags)]

test_data  = [{"tokens": t, "ner_tags": g}
              for t, g in zip(test_tokens, test_tags)]

print(f"\n DATASET OBJECTS ")
print(f"  train_data : {len(train_data)} examples")
print(f"  val_data   : {len(val_data)} examples")
print(f"  test_data  : {len(test_data)} examples  ← FROZEN")

 ENTITY COUNTS BEFORE OVERSAMPLING 
  TERM           :  2347  ==============================
  DATE           :  1188  ===============
  AMOUNT         :  1031  =============
  JURISDICTION   :   510  ======
  PARTY          :   486  ======

  Median count   : 1031
  Max multiplier : 2.0x

 OVERSAMPLING PLAN 
  PARTY          :  486 →  972  (2.0x)  OK
  DATE           : 1188 → 1188  (1.0x)  OK
  AMOUNT         : 1031 → 1031  (1.0x)  OK
  TERM           : 2347 → 2347  (1.0x)  OK
  JURISDICTION   :  510 → 1020  (2.0x)  OK

 ENTITY COUNTS AFTER OVERSAMPLING 
  TERM           :  2347  ==============================
  DATE           :  1188  ===============
  AMOUNT         :  1031  =============
  JURISDICTION   :  1020  =============
  PARTY          :   972  ============

  Total training examples: 6558

 DATASET OBJECTS 
  train_data : 6558 examples
  val_data   : 1192 examples
  test_data  : 1192 examples  ← FROZEN


In [27]:
total_entities = sum(after.values())
n_labels       = len(LABEL_LIST)

class_weights = []
print(" CLASS WEIGHTS")

for label in LABEL_LIST:
    if label == "O":
        w = 0.1
    elif label.startswith(("B-", "I-")):
        entity = label[2:]
        count  = after.get(entity, 1)
        w      = round(total_entities / (n_labels * count), 3)
    else:
        w = 1.0

    class_weights.append(w)
    bar = "=" * int(w * 20)
    print(f"  {label:22s}: {w:.3f}  {bar}")

print(f"\n  O weight        : {class_weights[0]:.3f}")
print(f"  B-JURISDICTION  : {class_weights[LABEL2ID['B-JURISDICTION']]:.3f}")
print(f"  B-PARTY         : {class_weights[LABEL2ID['B-PARTY']]:.3f}")
print(f"  B-TERM          : {class_weights[LABEL2ID['B-TERM']]:.3f}")
print(f"  Ratio JURIS/O   : {class_weights[LABEL2ID['B-JURISDICTION']]/class_weights[0]:.1f}x")

with open(f'{BASE}/src/class_weights.json', 'w') as f:
    json.dump(class_weights, f, indent=2)
print(f"\n class_weights.json saved")

 CLASS WEIGHTS
  O                     : 0.100  ==
  B-PARTY               : 0.613  ============
  I-PARTY               : 0.613  ============
  B-DATE                : 0.502  ==========
  I-DATE                : 0.502  ==========
  B-AMOUNT              : 0.578  ===========
  I-AMOUNT              : 0.578  ===========
  B-TERM                : 0.254  =====
  I-TERM                : 0.254  =====
  B-JURISDICTION        : 0.584  ===========
  I-JURISDICTION        : 0.584  ===========

  O weight        : 0.100
  B-JURISDICTION  : 0.584
  B-PARTY         : 0.613
  B-TERM          : 0.254
  Ratio JURIS/O   : 5.8x

 class_weights.json saved


In [28]:
!pip install transformers datasets -q

from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f" Tokenizer loaded — vocab size: {tokenizer.vocab_size:,}")

dataset = DatasetDict({
    "train"     : Dataset.from_dict({
                    "tokens"  : [d["tokens"]   for d in train_data],
                    "ner_tags": [d["ner_tags"]  for d in train_data]}),
    "validation": Dataset.from_dict({
                    "tokens"  : [d["tokens"]   for d in val_data],
                    "ner_tags": [d["ner_tags"]  for d in val_data]}),
    "test"      : Dataset.from_dict({
                    "tokens"  : [d["tokens"]   for d in test_data],
                    "ner_tags": [d["ner_tags"]  for d in test_data]}),
})
print(f"\n{dataset}")

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation          = True,
        max_length          = 320,
        is_split_into_words = True,
        padding             = False,
    )

    aligned_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids  = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word = None

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)
            prev_word = word_id

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

print("\nTokenizing all splits...")
tokenized = dataset.map(
    tokenize_and_align,
    batched        = True,
    remove_columns = ["tokens", "ner_tags"],
)

print(f"\n TOKENIZED DATASET ")
print(tokenized)

s = tokenized["train"][0]
print(f"\n  input_ids length : {len(s['input_ids'])}")
print(f"  labels length    : {len(s['labels'])}")
print(f"  lengths match    : {len(s['input_ids']) == len(s['labels'])} ")

print(f"\n ENTITY LABEL VERIFICATION ")
train_entities = set()
for example in tokenized["train"]:
    for l in example["labels"]:
        if l not in [-100, 0]:
            train_entities.add(ID2LABEL[l])

for entity in ["PARTY","DATE","AMOUNT","TERM","JURISDICTION"]:
    b_present = f"B-{entity}" in train_entities
    i_present = f"I-{entity}" in train_entities
    print(f"  {entity:15s}: B-{'OK' if b_present else 'NO'}  I-{'OK' if i_present else 'NO'}")

Loading tokenizer: nlpaueb/legal-bert-base-uncased


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

 Tokenizer loaded — vocab size: 30,522

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 6558
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1192
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1192
    })
})

Tokenizing all splits...


Map:   0%|          | 0/6558 [00:00<?, ? examples/s]

Map:   0%|          | 0/1192 [00:00<?, ? examples/s]

Map:   0%|          | 0/1192 [00:00<?, ? examples/s]


 TOKENIZED DATASET 
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 6558
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1192
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1192
    })
})

  input_ids length : 320
  labels length    : 320
  lengths match    : True 

 ENTITY LABEL VERIFICATION 
  PARTY          : B-OK  I-OK
  DATE           : B-OK  I-OK
  AMOUNT         : B-OK  I-OK
  TERM           : B-OK  I-OK
  JURISDICTION   : B-OK  I-OK


##Saving and Zipping

In [29]:
import os

TOKENIZED_SAVE = f'{BASE}/data/processed/tokenized_dataset_v2'
tokenized.save_to_disk(TOKENIZED_SAVE)
print(f"✅ Tokenized dataset saved → tokenized_dataset_v2/")

label_config_v2 = {
    "label_list": LABEL_LIST,
    "label2id"  : LABEL2ID,
    "id2label"  : {str(k): v for k, v in ID2LABEL.items()},
    "num_labels": len(LABEL_LIST)
}
with open(f'{BASE}/src/label_config.json', 'w') as f:
    json.dump(label_config_v2, f, indent=2)
print("✅ label_config.json saved")

with open(f'{BASE}/src/class_weights.json', 'w') as f:
    json.dump(class_weights, f, indent=2)
print("✅ class_weights.json saved")

with open(f'{BASE}/src/entity_map.json', 'w') as f:
    json.dump(ENTITY_MAP, f, indent=2)
print("✅ entity_map.json saved")

print(f"\n=== FINAL DATASET SIZES ===")
print(f"  Train      : {len(tokenized['train'])}")
print(f"  Validation : {len(tokenized['validation'])}")
print(f"  Test       : {len(tokenized['test'])}  ← FROZEN")

print(f"\n=== ALL FILES SAVED ===")
for fname in ['label_config.json', 'class_weights.json', 'entity_map.json']:
    path   = f'{BASE}/src/{fname}'
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} src/{fname}")

Saving the dataset (0/1 shards):   0%|          | 0/6558 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1192 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1192 [00:00<?, ? examples/s]

✅ Tokenized dataset saved → tokenized_dataset_v2/
✅ label_config.json saved
✅ class_weights.json saved
✅ entity_map.json saved

=== FINAL DATASET SIZES ===
  Train      : 6558
  Validation : 1192
  Test       : 1192  ← FROZEN

=== ALL FILES SAVED ===
  ✅ src/label_config.json
  ✅ src/class_weights.json
  ✅ src/entity_map.json


In [30]:
import shutil

print("\nZipping for Kaggle upload...")

shutil.make_archive('/content/tokenized_dataset_v2', 'zip',
                    f'{BASE}/data/processed', 'tokenized_dataset_v2')

shutil.make_archive('/content/src_configs', 'zip',
                    f'{BASE}', 'src')

import os
td_size = os.path.getsize('/content/tokenized_dataset_v2.zip') / 1024 / 1024
sc_size = os.path.getsize('/content/src_configs.zip')          / 1024 / 1024
print(f" tokenized_dataset_v2.zip : {td_size:.1f} MB")
print(f" src_configs.zip          : {sc_size:.1f} MB")

from google.colab import files
files.download('/content/tokenized_dataset_v2.zip')
files.download('/content/src_configs.zip')


Zipping for Kaggle upload...
 tokenized_dataset_v2.zip : 3.9 MB
 src_configs.zip          : 0.0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##test

In [14]:
print("\n=== LABEL DISTRIBUTION ===")
label_counts = Counter()

for ex in raw_data:
    for tag in ex["ner_tags"]:
        label_counts[ID2LABEL[tag]] += 1

for k,v in sorted(label_counts.items(), key=lambda x:-x[1]):
    print(f"{k:15s}: {v}")

total_tokens = sum(label_counts.values())
entity_tokens = total_tokens - label_counts["O"]

print("\n=== ENTITY TOKEN RATIO ===")
print(f"Total tokens   : {total_tokens}")
print(f"Entity tokens  : {entity_tokens}")
print(f"Entity percent : {entity_tokens/total_tokens*100:.2f}%")


=== LABEL DISTRIBUTION ===
O              : 2642109
I-TERM         : 120947
I-AMOUNT       : 73016
I-DATE         : 30309
I-JURISDICTION : 20166
I-PARTY        : 2009
B-TERM         : 1995
B-DATE         : 1466
B-AMOUNT       : 1150
B-JURISDICTION : 579
B-PARTY        : 527

=== ENTITY TOKEN RATIO ===
Total tokens   : 2894273
Entity tokens  : 252164
Entity percent : 8.71%
